# VoiceHub training: specialized TTS and ASR fine-tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/training.ipynb)

This notebook shows the shared training lifecycle while keeping architecture-specific objectives explicit. It includes codec/LLM TTS, diffusion/flow TTS, full adversarial VITS, and CTC ASR templates. Begin with one optimizer step and inspect gradients, losses, batches, and artifacts before scaling the run.

Every real checkpoint and training cell is opt-in. The profile-inspection cells are offline-safe.

## 0. Install the training environment

The single `training` extra covers every registered training adapter. The setup cell installs it only when VoiceHub is not already importable. Pin the code revision and record it with dataset and checkpoint revisions.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_VOICEHUB = importlib.util.find_spec("voicehub") is None
VOICEHUB_REVISION = "main"  # Prefer a release tag or full commit SHA.

if INSTALL_VOICEHUB:
    package = (
        "voicehub[training] @ "
        "git+https://github.com/kadirnar/voicehub.git@"
        f"{VOICEHUB_REVISION}"
    )
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        package,
    ])


In [ ]:
from pathlib import Path

RUN_CODEC_LM_TRAINING = False
RUN_DIFFUSION_TRAINING = False
RUN_VITS_TRAINING = False
RUN_ASR_TRAINING = False
RUN_TINY_TRAINER = False

DEVICE = "cuda"
MAX_STEPS = 1
RUNS_ROOT = Path("runs")
DATA_ROOT = Path("data")

active_trainer = None
active_model = None
active_run_name = None


## 1. Inspect the exact training profile

`ModelTrainingSpec` declares the objective family, support boundary, phase schedule, optimizer ownership, and dataset contract without loading a model. Do not infer training behavior from an inference architecture name alone.

In [ ]:
from voicehub import SpeechTask, get_training_spec, list_training_specs

SPECIALIZED_PROFILES = {
    "codec_llm": "conversationtts",
    "diffusion_flow": "f5tts",
    "vits_gan": "vits",
    "asr_ctc": "asr_wav2vec2",
    "asr_seq2seq": "asr_whisper",
    "asr_transducer": "asr_nemotron",
}

profile_summary = {}
for label, model_type in SPECIALIZED_PROFILES.items():
    profile = get_training_spec(model_type)
    dataset_spec = profile.dataset_spec
    profile_summary[label] = {
        "model_type": model_type,
        "family": profile.family_name,
        "support": profile.support.value,
        "recipe": profile.recipe_kind.value,
        "separate_optimizers": profile.separate_optimizers,
        "phases": tuple(phase.name for phase in profile.phases),
        "data_architecture": dataset_spec.architecture.value,
        "data_readiness": dataset_spec.readiness.value,
    }
    print(label, profile_summary[label])

asr_training_profiles = tuple(
    spec.model_type
    for spec in list_training_specs(
        task=SpeechTask.AUTOMATIC_SPEECH_RECOGNITION,
    )
)
print("All trainable ASR profiles:", len(asr_training_profiles))
print(", ".join(asr_training_profiles))


## 2. Start with one explicit optimizer step

The same `Trainer` and `TrainingArguments` surface orchestrates every recipe. The selected model adapter still owns objective computation, component freezing, phase ordering, and named optimizer routing.

In [ ]:
from voicehub import TrainingArguments


def make_training_arguments(run_name, *, learning_rate):
    return TrainingArguments(
        output_dir=str(RUNS_ROOT / run_name),
        max_steps=MAX_STEPS,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=learning_rate,
        logging_steps=1,
        save_steps=1,
        save_total_limit=2,
        eval_strategy="no",
        report_to=[],
        use_cpu=DEVICE == "cpu",
        seed=42,
    )


smoke_arguments = make_training_arguments(
    "one-step-smoke",
    learning_rate=1e-5,
)
print(smoke_arguments.output_dir, smoke_arguments.max_steps)


## 3. Verify the Trainer offline

Before downloading a speech checkpoint, optionally run two CPU steps through the real `Trainer` with a tiny differentiable module. This checks batching, backward, optimizer stepping, and metric reporting without network access.

In [ ]:
if RUN_TINY_TRAINER:
    import torch

    from voicehub import TTSTrainingOutput, Trainer

    torch.manual_seed(42)

    class TinySpeechRegressor(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.projection = torch.nn.Linear(1, 1)

        def forward(self, input_values, labels=None):
            logits = self.projection(input_values.float())
            loss = None
            if labels is not None:
                loss = torch.nn.functional.mse_loss(
                    logits,
                    labels.float(),
                )
            return TTSTrainingOutput(loss=loss, logits=logits)

    tiny_rows = [
        {
            "input_values": torch.tensor([float(index)]),
            "labels": torch.tensor([float(2 * index)]),
        }
        for index in range(4)
    ]
    tiny_trainer = Trainer(
        model=TinySpeechRegressor(),
        args=TrainingArguments(
            output_dir=str(RUNS_ROOT / "tiny-trainer-smoke"),
            max_steps=2,
            per_device_train_batch_size=2,
            learning_rate=1e-2,
            logging_strategy="no",
            save_strategy="no",
            report_to="none",
            use_cpu=True,
            seed=42,
            data_seed=42,
        ),
        train_dataset=tiny_rows,
    )
    tiny_output = tiny_trainer.train()
    assert tiny_output.global_step == 2
    print(tiny_output)
else:
    print("Tiny Trainer smoke test is disabled.")


## 4. Understand specialized objective boundaries

These small tensor examples expose the strict primitives used by architecture adapters. Real integrations additionally own their tokenizers, codecs, latent extractors, schedulers, discriminators, masking, EMA, and checkpoint semantics.

In [ ]:
import torch

from voicehub import (
    build_flow_matching_training_pair,
    masked_diffusion_regression_loss,
    multi_codebook_cross_entropy,
    vits_discriminator_loss,
    vits_feature_matching_loss,
    vits_generator_adversarial_loss,
    vits_kl_loss,
)

torch.manual_seed(42)

# Codec/LLM: preserve batch, codebook, time, and vocabulary axes.
codec_logits = torch.randn(1, 2, 4, 8, requires_grad=True)
codec_labels = torch.randint(0, 8, (1, 2, 4))
codec_mask = torch.ones_like(codec_labels, dtype=torch.bool)
codec_loss = multi_codebook_cross_entropy(
    codec_logits,
    codec_labels,
    loss_mask=codec_mask,
    causal_shift=True,
    sequence_dim=2,
    codebook_dim=1,
)

# Flow matching: sample time/noise during training and regress velocity.
clean_latents = torch.randn(2, 6, 4)
flow_pair = build_flow_matching_training_pair(
    clean_latents,
    generator=torch.Generator().manual_seed(42),
)
flow_prediction = torch.zeros_like(flow_pair.targets, requires_grad=True)
flow_loss = masked_diffusion_regression_loss(
    flow_prediction,
    flow_pair.targets,
    mask=torch.ones(2, 6, dtype=torch.bool),
)

# VITS/GAN: discriminator, generator, feature, and KL terms stay distinct.
real_scores = [torch.full((1, 3), 0.8, requires_grad=True)]
discriminator_fake_scores = [
    torch.full((1, 3), 0.2, requires_grad=True)
]
generator_fake_scores = [
    torch.full((1, 3), 0.2, requires_grad=True)
]
discriminator_loss = vits_discriminator_loss(
    real_scores,
    discriminator_fake_scores,
).loss
generator_loss = vits_generator_adversarial_loss(generator_fake_scores)
real_features = [[torch.zeros(1, 2, requires_grad=True)]]
fake_features = [[torch.ones(1, 2, requires_grad=True)]]
feature_loss = vits_feature_matching_loss(real_features, fake_features)
posterior = torch.full((1, 2, 4), 2.0, requires_grad=True)
zeros = torch.zeros_like(posterior)
kl_loss = vits_kl_loss(posterior, zeros, zeros, zeros)

print({
    "codec": float(codec_loss.detach()),
    "flow": float(flow_loss.detach()),
    "vits_discriminator": float(discriminator_loss.detach()),
    "vits_generator": float(generator_loss.detach()),
    "vits_features": float(feature_loss.detach()),
    "vits_kl": float(kl_loss.detach()),
})


## 5. Codec/LLM TTS fine-tuning

ConversationTTS accepts raw text/audio or prepared text and Mimi codebooks. Its native adapter constructs the published text/audio framing and two-level objective. The codec target path remains model-owned.

In [ ]:
if RUN_CODEC_LM_TRAINING:
    from voicehub import (
        AutoModelForTextToSpeech,
        TTSDataset,
        Trainer,
        load_audio,
    )

    active_run_name = "conversationtts-domain"
    active_model = AutoModelForTextToSpeech.from_pretrained(
        "AudioFoundation/SpeechFoundation",
        model_type="conversationtts",
        device=DEVICE,
        lazy_load=True,
    )
    active_model.validate_training_support()
    source = TTSDataset.from_manifest(
        DATA_ROOT / "conversationtts.jsonl",
        model_type="conversationtts",
        validate_files=True,
    )
    source_train, source_eval = source.train_test_split(
        validation_fraction=0.1,
        seed=42,
        group_by="speaker_id",
    )
    active_trainer = Trainer(
        model=active_model,
        args=make_training_arguments(active_run_name, learning_rate=1e-5),
        train_dataset=active_model.create_training_dataset(source_train),
        eval_dataset=active_model.create_training_dataset(source_eval),
    )
    active_trainer.train()
else:
    print("Codec/LLM training is disabled.")


## 6. Diffusion/flow TTS fine-tuning

F5-TTS uses the released conditional-flow objective. Its dataset boundary is deliberately preprocessed: provide 24 kHz waveforms with checkpoint vocabulary IDs, or 100-bin mel frames with the same text IDs. Vocos remains frozen.

In [ ]:
if RUN_DIFFUSION_TRAINING:
    import torch

    from voicehub import (
        AutoModelForTextToSpeech,
        TTSDataset,
        Trainer,
        load_audio,
    )

    active_run_name = "f5tts-domain"
    active_model = AutoModelForTextToSpeech.from_pretrained(
        "F5TTS_v1_Base",
        model_type="f5tts",
        device=DEVICE,
        lazy_load=True,
    )
    active_model.validate_training_support()

    # Replace these IDs with the selected checkpoint vocabulary encoding.
    token_ids = torch.tensor([12, 31, 7, 4], dtype=torch.long)
    f5_records = []
    for audio_name in ("flow-001.wav", "flow-002.wav"):
        audio = load_audio(
            DATA_ROOT / "f5tts" / audio_name,
            target_sampling_rate=24_000,
        )
        f5_records.append({
            "input_values": audio.waveform,
            "input_ids": token_ids,
        })
    source = TTSDataset(f5_records, model_type="f5tts")
    source_train, source_eval = source.train_test_split(
        validation_fraction=0.5,
        seed=42,
    )
    active_trainer = Trainer(
        model=active_model,
        args=make_training_arguments(active_run_name, learning_rate=1e-5),
        train_dataset=active_model.create_training_dataset(source_train),
        eval_dataset=active_model.create_training_dataset(source_eval),
    )
    active_trainer.train()
else:
    print("Diffusion/flow training is disabled.")


## 7. Full adversarial VITS fine-tuning

VITS uses independently optimized discriminator and generator phases. The original MMS-TTS metadata omits its source acoustic settings, so the exact FFT, hop, window, mel, and segment configuration must come from the checkpoint's source training recipe rather than being guessed.

In [ ]:
if RUN_VITS_TRAINING:
    from voicehub import AutoModelForTextToSpeech, TTSDataset, Trainer

    active_run_name = "vits-domain"
    active_model = AutoModelForTextToSpeech.from_pretrained(
        "facebook/mms-tts-eng",
        model_type="vits",
        device=DEVICE,
        lazy_load=True,
        enable_native_adversarial_training=True,
        training_acoustic_config={
            # Verify every value against the selected source recipe.
            "sampling_rate": 16_000,
            "filter_length": 1_024,
            "hop_length": 256,
            "win_length": 1_024,
            "num_mel_channels": 80,
            "mel_fmin": 0.0,
            "mel_fmax": 8_000.0,
            "segment_size": 8_192,
        },
    )
    active_model.validate_training_support()
    def decode_vits_audio(record):
        prepared = dict(record)
        decoded = load_audio(
            prepared.pop("audio"),
            target_sampling_rate=16_000,
        )
        prepared["audio_values"] = decoded.waveform
        prepared["sampling_rate"] = decoded.sampling_rate
        return prepared

    source = TTSDataset.from_manifest(
        DATA_ROOT / "vits.jsonl",
        model_type="vits",
        validate_files=True,
        transform=decode_vits_audio,
        transform_fingerprint="vits-pcm-16khz-v1",
    )
    source_train, source_eval = source.train_test_split(
        validation_fraction=0.1,
        seed=42,
        group_by="speaker_id",
    )
    active_trainer = Trainer(
        model=active_model,
        args=make_training_arguments(active_run_name, learning_rate=2e-4),
        train_dataset=active_model.create_training_dataset(source_train),
        eval_dataset=active_model.create_training_dataset(source_eval),
    )
    active_trainer.train()
else:
    print("VITS adversarial training is disabled.")


## 8. ASR fine-tuning

The same lifecycle covers CTC, speech sequence-to-sequence, RNN-T, TDT, and hybrid CTC/attention providers. The model adapter preserves tokenizer, blank/alignment, prompt, duration, and native objective semantics.

In [ ]:
if RUN_ASR_TRAINING:
    from voicehub import ASRDataset, AutoModelForSpeechRecognition, Trainer

    active_run_name = "wav2vec2-domain"
    active_model = AutoModelForSpeechRecognition.from_pretrained(
        "facebook/wav2vec2-base-960h",
        model_type="asr_wav2vec2",
        device=DEVICE,
        lazy_load=True,
    )
    active_model.validate_training_support()
    source = ASRDataset.from_manifest(
        DATA_ROOT / "asr.jsonl",
        model_type="asr_wav2vec2",
        validate_files=True,
    )
    source_train, source_eval = source.train_test_split(
        validation_fraction=0.1,
        seed=42,
        group_by="speaker_id",
    )
    active_trainer = Trainer(
        model=active_model,
        args=make_training_arguments(active_run_name, learning_rate=3e-5),
        train_dataset=active_model.create_training_dataset(source_train),
        eval_dataset=active_model.create_training_dataset(source_eval),
    )
    active_trainer.train()
else:
    print("ASR training is disabled.")


## 9. Evaluate, resume, and export

Periodic `checkpoint-N/` directories retain optimizer, scheduler, RNG, sampler, phase, and callback state for exact continuation. `save_model()` writes a portable inference/warm-start artifact; it is not a replacement for a complete resume checkpoint.

In [ ]:
if active_trainer is not None:
    metrics = active_trainer.evaluate()
    final_directory = RUNS_ROOT / active_run_name / "final"
    active_trainer.save_model(final_directory)
    print(metrics)
    print("Portable artifact:", final_directory)

    # After interruption, use the same model, recipe, data order, and args:
    # active_trainer.train(resume_from_checkpoint=True)
else:
    print("Run one architecture section before exporting an artifact.")


## Scaling checklist

1. Verify the checkpoint license and exact training support boundary.
2. Freeze an auditable, group-disjoint dataset split and fingerprint.
3. Inspect one model-owned batch, masks, label ranges, and trainable positions.
4. Run one step and confirm finite loss, expected gradients, and phase-local optimizer ownership.
5. Reload an exact checkpoint and compare the next step deterministically.
6. Export a fresh portable artifact and run fixed-prompt inference.
7. Increase steps, batch size, precision, and execution strategy only after the smoke run passes.

See the [training guide](https://kadirnar.github.io/voicehub/guides/training/) and [training support matrix](https://kadirnar.github.io/voicehub/models/training-support/) for model-specific boundaries.